# Estimation of 3SLS

Rの標準関数に加えて、`systemfit` パッケージを使用します。

In [36]:
if (!require(systemfit)) install.packages("systemfit")
library(systemfit)

In [37]:
# CSVファイルの読み込み
data <- read.csv("../data/8-sem.csv")

In [38]:
# データを確認（先頭5件）
head(data, 5)

,Year,GDP,I,GOV,M,TB
,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,1970,1035.6,150.2,115.9,628.1,6.562
2,1971,1125.4,176.0,117.1,712.7,4.511
3,1972,1237.3,205.6,125.1,805.2,4.466
4,1973,1382.6,242.9,128.2,861.0,7.178
5,1974,1496.9,245.6,139.9,908.5,7.926


In [39]:
# ラグ変数の作成
data <- data[order(data$Year), ]
data$L_GDP <- c(NA, head(data$GDP, -1))
data$L_M   <- c(NA, head(data$M, -1))
data$L_I  <- c(NA, head(data$I, -1))
data$L_GOV <- c(NA, head(data$GOV, -1))
data$L_TB  <- c(NA, head(data$TB, -1))

In [40]:
# 各方程式の定義
eq1 <- GDP ~ M + I + GOV
eq2 <- M ~ GDP + TB
eqSystem <- list(GDPeq = eq1, Meq = eq2)
inst_form <- ~ I + GOV + TB + L_GDP + L_M + L_I + L_GOV + L_TB

In [41]:
# 3SLSの推定
fit3sls <- systemfit(eqSystem,
                     method = "3SLS",
                     inst = inst_form,
                     data = data)

In [42]:
fit3sls


systemfit results 
method: 3SLS 

Coefficients:
GDPeq_(Intercept)           GDPeq_M           GDPeq_I         GDPeq_GOV 
     -236.9870115         2.5636649        -0.0542271        -4.9947808 
  Meq_(Intercept)           Meq_GDP            Meq_TB 
      100.4839793         0.5534168         6.0332419 

In [44]:
summary(fit3sls)


systemfit results 
method: 3SLS 

        N DF     SSR  detRCov   OLS-R2 McElroy-R2
system 48 41 1179726 43376416 0.988546   0.999599

       N DF    SSR     MSE    RMSE       R2   Adj R2
GDPeq 24 20 843933 42196.7 205.418 0.989305 0.987701
Meq   24 21 335792 15990.1 126.452 0.986061 0.984734

The covariance matrix of the residuals used for estimation
         GDPeq      Meq
GDPeq  37217.9 -18119.1
Meq   -18119.1  15953.3

The covariance matrix of the residuals
         GDPeq      Meq
GDPeq  42196.7 -25126.7
Meq   -25126.7  15990.1

The correlations of the residuals
          GDPeq       Meq
GDPeq  1.000000 -0.967323
Meq   -0.967323  1.000000


3SLS estimates for 'GDPeq' (equation 1)
Model Formula: GDP ~ M + I + GOV
Instruments: ~I + GOV + TB + L_GDP + L_M + L_I + L_GOV + L_TB

                Estimate   Std. Error  t value   Pr(>|t|)    
(Intercept) -236.9870115   97.7670860 -2.42400   0.024954 *  
M              2.5636649    0.4006720  6.39841 3.0512e-06 ***
I             -0.0542271